# MLIP Structure Relaxation with AiiDA-PythonJob

This notebook demonstrates how to perform structure relaxation using Machine Learning Interatomic Potentials (MLIPs) through `aiida-pythonjob` and `aiida-muon`.

## Requirements
- `aiida-core`
- `aiida-pythonjob`
- `aiida-muon`
- MLIP package (e.g., `mace-torch`, `chgnet`, `m3gnet`)
- `ase`
- `pymatgen`

In [1]:
from aiida import load_profile, orm
from aiida.engine import submit, run
from aiida_pythonjob import PythonJob
from pymatgen.io.cif import CifParser
import numpy as np

load_profile()

Profile<uuid='dd482d6b5e0344d1a9ba01ea32d227fe' name='default'>

## 1. Load and Visualize Structure

Let's start by loading a simple structure (Silicon).

In [2]:
# Load structure from CIF
parser = CifParser("/home/jovyan/bind_mount/codes/aiida-muon/examples/data/LaCoPO_bcs_file_26735.mcif")
structure_pmg = parser.get_structures(primitive=True)[0]
structure = orm.StructureData(pymatgen=structure_pmg)

print(f"Formula: {structure_pmg.formula}")
print(f"Number of atoms: {len(structure_pmg)}")
print(f"Lattice parameters: {structure_pmg.lattice.abc}")
print(f"Space group: {structure_pmg.get_space_group_info()}")

/tmp/ipykernel_3990234/2982467415.py:3: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structure_pmg = parser.get_structures(primitive=True)[0]
/home/jovyan/.conda/envs/base_311/lib/python3.11/site-packages/pymatgen/io/cif.py:1349: UserWarning: Issues encountered while parsing CIF: Skipping relative stoichiometry check because CIF does not contain formula keys.
  return self.parse_structures(*args, **kwargs)


Formula: La2 Co2 P2 O2
Number of atoms: 8
Lattice parameters: (3.966, 3.966, 8.365)
Space group: ('P4/nmm', 129)


## 2. Setup MLIP Calculator

We'll use MACE-MP (Materials Project) as our MLIP calculator. The calculator must be defined as a function that will be executed remotely.

In [3]:
def get_mace_calculator():
    """Create and return a MACE calculator.
    
    This function will be executed on the remote computer.
    All imports must be inside the function.
    """
    from mace.calculators import mace_mp
    return mace_mp(
        model="medium",      # Options: small, medium, large
        device="cpu",        # Use 'cuda' if GPU available
        default_dtype="float64",
        dispersion=False     # Set True to include dispersion corrections
    )

def get_mattersim_calculator():
    """Create and return a Mattersim calculator.
    
    This function will be executed on the remote computer.
    All imports must be inside the function.
    
    Note: Works around pkg_resources import in mattersim.__version__.
    """
    import sys
    
    # Workaround for missing pkg_resources: patch mattersim.__version__ module
    # This prevents the import error when mattersim tries to import pkg_resources
    class FakeVersion:
        __version__ = "1.0.0"  # Dummy version
    
    # Pre-create the __version__ module to avoid pkg_resources import
    import types
    version_module = types.ModuleType('mattersim.__version__')
    version_module.__version__ = "1.0.0"
    sys.modules['mattersim.__version__'] = version_module
    
    from mattersim.forcefield import MatterSimCalculator
    mattersim_calculator = MatterSimCalculator(
        load_path="/home/jovyan/bind_mount/codes/mattersim/pretrained_models/mattersim-v1.0.0-5M.pth",
        device="cpu",
    )
    return mattersim_calculator

def get_nequip_calculator():
    """Create and return a NEquIP calculator.
    
    This function will be executed on the remote computer.
    All imports must be inside the function.
    
    Note: Handles PyTorch 2.6 weights_only compatibility issue with e3nn.
    """
    # Fix for PyTorch 2.6: Allow slice in safe globals for e3nn's constants.pt
    import torch
    torch.serialization.add_safe_globals([slice])
    
    from nequip.ase import NequIPCalculator
    nequip_calculator = NequIPCalculator.from_compiled_model(
        compile_path="/home/jovyan/bind_mount/codes/compiled_mp_l_01.nequip.pt2",
        device="cpu",
    )
    return nequip_calculator

mlips = ['mace', 'mattersim', 'nequip']

mlip = mlips[1] 

# Load your configured PythonJob code
# If not configured, run: verdi code create core.code.installed ...
if mlip == 'mace':
    pythonjob_code = orm.load_code('python3@localhost')
    callback_calculator = get_mace_calculator
elif mlip == 'mattersim':
    pythonjob_code = orm.load_code('python3_mattersim_p311@localhost')
    callback_calculator = get_mattersim_calculator
elif mlip == 'nequip':
    pythonjob_code = orm.load_code('python3_nequip_p311@localhost')
    callback_calculator = get_nequip_calculator

pythonjob_metadata = {
    'label': f'{mlip}_relaxation',
    'description': 'Silicon relaxation using MACE-MP medium',
    'options': {
        'resources': {'num_machines': 1, 'num_mpiprocs_per_machine': 1},
        'max_wallclock_seconds': 1800,
    }
}


## 3. Prepare PythonJob Inputs

Use the utility function to prepare all necessary inputs for the relaxation job.

In [4]:
from aiida_muon.pythonjobs.relax import prepare_pythonjob_inputs

In [5]:
pythonjob_inputs = prepare_pythonjob_inputs(
    structure=structure,
    pythonjob_code=pythonjob_code,
    callback_calculator=callback_calculator,
#    pythonjob_metadata=pythonjob_metadata,
#    fix_symmetry=True
#     optimizer='BFGS',
#     trajectory='ase.traj',
#     fmax=1e-4
)

In [6]:
pythonjob_inputs

{'function_data': {'name': 'relax_function',
  'source_code': '    def relax_function(atoms, fmax=1e-4, optimizer=\'BFGS\', trajectory=None, \n                   fix_symmetry=False, optimizer_kwargs=None):\n        """Convenience function for ASE-based relaxation that returns the optimized structure.\n        \n        This is the main function to be called from aiida-pythonjob for structure relaxations\n        using any ASE calculator (MLIPs, DFT codes, empirical potentials, etc.).\n        """\n\n        result = optimize_function(\n            atoms, calculator=callback_calculator, fmax=fmax, optimizer=optimizer, \n            trajectory=trajectory, optimizer_kwargs=optimizer_kwargs,\n            fix_symmetry=fix_symmetry\n        )\n        return {\n            "structure": result[\'structure\'], \n            "energy": result[\'energy\'], \n            "forces": result[\'forces\'], \n            "nsteps": result[\'nsteps\']\n        }\n',
  'mode': 'use_pickled_function',
  'pic

## 4. Submit the Calculation

Submit the relaxation job to the AiiDA daemon.

In [7]:
# Submit the job
calc_node = submit(PythonJob, **pythonjob_inputs)

print(f"Submitted PythonJob: PK={calc_node.pk}")
print(f"UUID: {calc_node.uuid}")
print(f"\nMonitor with: verdi process show {calc_node.pk}")
print(f"Watch live: verdi process watch {calc_node.pk}")

Submitted PythonJob: PK=79049
UUID: ac876be4-5acf-44c1-8973-f816ba898fda

Monitor with: verdi process show 79049
Watch live: verdi process watch 79049


## 5. Wait for Completion and Analyze Results

Wait for the calculation to complete and then analyze the relaxed structure.

In [8]:
# Wait for calculation (or check manually with verdi process list)
import time

print("Waiting for calculation to complete...")
while not calc_node.is_terminated:
    time.sleep(5)
    print(f"Status: {calc_node.process_state}")

print(f"\nFinal status: {calc_node.process_state}")
print(f"Is finished ok: {calc_node.is_finished_ok}")

Waiting for calculation to complete...
Status: ProcessState.WAITING
Status: ProcessState.WAITING
Status: ProcessState.FINISHED

Final status: ProcessState.FINISHED
Is finished ok: True


In [10]:
calc_node.outputs.structure , calc_node.outputs.energy.value, calc_node.outputs.forces.get_array("default"), calc_node.outputs.nsteps.value

(<StructureData: uuid: b877c4ec-1cd2-48b1-9f6f-d344d29f0817 (pk: 79056)>,
 -55.011013031006,
 array([[ 6.7542123e-07,  1.9200756e-06,  2.0124987e-03],
        [-5.0570816e-06,  4.8428387e-07, -2.0182133e-03],
        [-7.2235707e-07,  3.6946108e-06,  9.5360861e-07],
        [-4.5483321e-06,  3.3131801e-07,  2.2176823e-06],
        [ 1.9611855e-06, -3.4694665e-09,  1.8434227e-04],
        [ 6.2148320e-06, -4.7389794e-06, -1.8137693e-04],
        [-1.7965212e-06, -9.7975135e-07, -1.4015161e-06],
        [ 3.2777898e-06, -7.6182187e-07,  1.1388132e-06]], dtype=float32),
 4)

In [ ]:
# Get the relaxed structure
if calc_node.is_finished_ok:
    
    print("Relaxed structure:")
    print(f"  Final energy: {calc_node.outputs.energy.value:.6f} eV")
    print(f"  Final positions:\n{calc_node.outputs.structure.value.get_positions()}")
    print(f"  Final forces (max): {np.abs(calc_node.outputs.forces.get_array('default')).max():.6f} eV/Å")
    
else:
    print("Calculation failed!")
    print(f"Exit status: {calc_node.exit_status}")

Relaxed structure:
  Final energy: -55.010998 eV
  Final positions:
[[ 9.91500402e-01  9.91499953e-01  7.07716093e+00]
 [ 2.97449952e+00  2.97449995e+00  1.28783981e+00]
 [ 2.97450018e+00  9.91500023e-01  4.18249979e+00]
 [ 9.91500032e-01  2.97450019e+00  4.18249976e+00]
 [ 9.91499943e-01  9.91499922e-01  3.18411468e+00]
 [ 2.97449996e+00  2.97449991e+00  5.18088523e+00]
 [ 2.97449964e+00  9.91499976e-01 -1.61203466e-07]
 [ 9.91500318e-01  2.97450008e+00 -7.49648072e-08]]
  Final forces (max): 0.002070 eV/Å


## 6. Compare Initial and Relaxed Structures

In [ ]:
if calc_node.is_finished_ok:
    initial_atoms = structure.get_ase()
    
    print("Comparison:")
    print(f"  Initial lattice: {initial_atoms.cell.cellpar()[:3]}")
    print(f"  Relaxed lattice: {relaxed_atoms.cell.cellpar()[:3]}")
    print(f"\n  Lattice change: {np.abs(relaxed_atoms.cell.cellpar()[:3] - initial_atoms.cell.cellpar()[:3])}")
    
    # Position changes
    pos_diff = relaxed_atoms.get_positions() - initial_atoms.get_positions()
    max_displacement = np.max(np.linalg.norm(pos_diff, axis=1))
    print(f"  Maximum atomic displacement: {max_displacement:.6f} Å")

Comparison:
  Initial lattice: [3.966 3.966 8.365]


NameError: name 'relaxed_atoms' is not defined

## 7. Example with Different MLIP: CHGNet

Let's try the same relaxation with a different MLIP (CHGNet).

In [ ]:
def get_chgnet_calculator():
    """Create CHGNet calculator."""
    from chgnet.model import CHGNet
    model = CHGNet.load()
    return model.get_calculator()

# Prepare and submit
chgnet_inputs = prepare_ase_relaxation_inputs(
    structure=structure,
    calculator=get_chgnet_calculator,
    fmax=1e-4,
    pythonjob_inputs={
        'code': pythonjob_code,
        'metadata': {
            'label': 'CHGNet_Si_relaxation',
            'description': 'Silicon relaxation using CHGNet',
        }
    }
)

chgnet_node = submit(PythonJob, **chgnet_inputs)
print(f"Submitted CHGNet calculation: PK={chgnet_node.pk}")

AttributeError: copy